# 원하는 포즈로 이미지 만드는 도구 (Pose → Image)

참조 사진의 **자세를 그대로 따라 하는 새 이미지**를 만드는 튜토리얼입니다.

1. 참조 사람 사진에서 **OpenPose**가 관절(스켈레톤)을 뽑는다
2. 그 포즈를 **ControlNet 조건**으로 넣어 **Stable Diffusion 1.5**가 같은 자세의 다른 인물/장면을 만든다

> 모델: SD1.5 + ControlNet(OpenPose). 무료 Colab **T4 GPU**에서 돌고, **고정 시드(seed=42)** 로 재현됩니다.
> 메뉴 > 런타임 > 런타임 유형 변경 > **T4 GPU** 로 바꾸고 위에서부터 순서대로 실행하세요.

## 1. 패키지 설치 + 환경 준비

In [ ]:
# 이 셀이 하는 일: 필요한 라이브러리 설치 + mediapipe 호환 처리
# (controlnet_aux가 mediapipe를 import하는데 Colab 파이썬 3.13과 버전 충돌 → 무해한 더미로 대체)
!pip -q install -U diffusers transformers accelerate controlnet_aux mediapipe
import sys, types, numpy as np
from unittest.mock import MagicMock
_mp = types.ModuleType('mediapipe'); _mp.solutions = MagicMock(); sys.modules['mediapipe'] = _mp

## 2. import + 포즈 추출기 로드

In [ ]:
# 이 셀이 하는 일: 도구 import + OpenPose 포즈 추출기 로드
import torch, os
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image
from controlnet_aux import OpenposeDetector
os.makedirs('samples', exist_ok=True)
openpose = OpenposeDetector.from_pretrained('lllyasviel/Annotators')
print('포즈 추출기 준비 완료')

## 3. 참조 사진에서 포즈 뽑기
예시로 인물 사진 URL을 씁니다. 내 사진을 쓰려면 왼쪽 파일탭에 업로드 후 `load_image('내파일.jpg')` 로 바꾸세요.
(추출된 스켈레톤이 너무 비면 준비된 전신 스켈레톤으로 자동 대체하는 안전장치 포함)

In [ ]:
# 이 셀이 하는 일: 참조 사진 읽기 -> OpenPose로 포즈(스켈레톤) 추출 -> 저장
photo = load_image('https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/input_image_vermeer.png')
pose = openpose(photo)
# 안전장치: 추출 스켈레톤이 거의 비면 준비된 전신 포즈로 대체
if (np.array(pose.convert('RGB')).sum(2) > 20).mean() < 0.003:
    pose = load_image('https://huggingface.co/datasets/hf-internal-testing/diffusers-images/resolve/main/sd_controlnet/pose.png')
pose.save('samples/pose_01.png')
pose   # 뽑힌 포즈 스켈레톤 확인

## 4. SD1.5 + ControlNet(OpenPose) 파이프라인 로드

In [ ]:
# 이 셀이 하는 일: 포즈 ControlNet + SD1.5 본체를 불러 파이프라인 구성
controlnet = ControlNetModel.from_pretrained('lllyasviel/sd-controlnet-openpose', torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    controlnet=controlnet, torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to('cuda')
print('파이프라인 준비 완료')

## 5. 생성 함수 (고정 시드 = 재현성)
seed를 고정해서 같은 입력이면 **항상 같은 결과**가 나오게 합니다 (루브릭: 재현성).

In [ ]:
# 이 셀이 하는 일: (포즈, 프롬프트) -> 생성 이미지 함수 정의
def generate(pose_img, prompt, seed=42):
    g = torch.Generator('cuda').manual_seed(seed)
    return pipe(prompt, image=pose_img, num_inference_steps=25, generator=g,
                negative_prompt='lowres, bad anatomy, worst quality, blurry').images[0]

## 6. 실행 — 같은 포즈 + 프롬프트 A

In [ ]:
# 이 셀이 하는 일: 포즈에 프롬프트 A(우주비행사) 로 생성 -> 저장
out1 = generate(pose, 'a photo of an astronaut in a white spacesuit, studio lighting, highly detailed')
out1.save('samples/output_01.png')
out1

## 7. 조건 바꿔보기 — 같은 포즈, 다른 프롬프트 B

In [ ]:
# 이 셀이 하는 일: 같은 포즈에 프롬프트 B(기사) 로 생성 -> 저장 (자세 유지 확인)
out2 = generate(pose, 'a medieval knight in shining steel armor, dramatic cinematic lighting, highly detailed')
out2.save('samples/output_02.png')
out2

## 8. 관찰 정리
- **같은 포즈 + 프롬프트만 변경(우주비행사→기사):** 두 결과 모두 **상반신 초상 · 고개를 옆으로 돌린 같은 자세**가 유지되고, 인물/의상만 우주비행사↔기사로 바뀜. → 프롬프트를 바꿔도 **입력 포즈는 그대로 반영**됨.
- **포즈 반영(루브릭 핵심):** 입력 스켈레톤(머리 기울임·어깨·팔)이 두 출력의 구도·머리 방향에 그대로 나타남.
- **재현성:** `manual_seed(42)` 고정 → 같은 프롬프트·포즈로 다시 실행하면 동일 이미지 재현.
- **한계:** 참조가 상반신 초상이라 하반신 포즈는 제어되지 않음. 전신 포즈를 원하면 전신 사진을 입력으로 쓰면 됨. 손가락 등 미세 부위는 OpenPose 정밀도 한계로 덜 정확.